In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Carica i dati 
df = pd.read_csv('C:/airflow-project/dataset.csv', sep=None, engine='python')
desc = pd.read_csv('C:/airflow-project/data_descriptions.csv', sep=';')
df.head()

,rev_Mean,mou_Mean,totmrc_Mean,da_Mean,ovrmou_Mean,ovrrev_Mean,vceovr_Mean,datovr_Mean,roam_Mean,change_mou,...,forgntvl,ethnic,kid0_2,kid3_5,kid6_10,kid11_15,kid16_17,creditcd,eqpdays,Customer_ID
0,"23,9975","219,25","22,5","0,2475",0,0,0,0,0,"-157,25",...,0.0,N,U,U,U,U,U,Y,361.0,1000001
1,"57,4925","482,75","37,425","0,2475","22,75","9,1","9,1",0,0,"532,25",...,0.0,Z,U,U,U,U,U,Y,240.0,1000002
2,"16,99","10,25","16,99",0,0,0,0,0,0,"-4,25",...,0.0,N,U,Y,U,U,U,Y,1504.0,1000003
3,38,"7,5",38,0,0,0,0,0,0,"-1,5",...,0.0,U,Y,U,U,U,U,Y,1812.0,1000004
4,"55,23","570,5","71,98",0,0,0,0,0,0,"38,5",...,0.0,I,U,U,U,U,U,Y,434.0,1000005


In [18]:
# 1. Mappa variabili (nome -> significato)
mapping = dict(zip(desc['Variable'], desc['Significado']))

In [19]:
# 2. Controllo tipi
print(df.dtypes)
print(df.select_dtypes(include='object').columns)  # categoriche
print(df.select_dtypes(include='number').columns)  # numeriche

rev_Mean        object
mou_Mean        object
totmrc_Mean     object
da_Mean         object
ovrmou_Mean     object
                ...   
kid11_15        object
kid16_17        object
creditcd        object
eqpdays        float64
Customer_ID      int64
Length: 100, dtype: object
Index(['rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean',
       'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou',
       'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean',
       'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean',
       'plcd_dat_Mean', 'recv_vce_Mean', 'recv_sms_Mean', 'comp_vce_Mean',
       'comp_dat_Mean', 'custcare_Mean', 'ccrndmou_Mean', 'cc_mou_Mean',
       'inonemin_Mean', 'threeway_Mean', 'mou_cvce_Mean', 'mou_cdat_Mean',
       'mou_rvce_Mean', 'owylis_vce_Mean', 'mouowylisv_Mean',
       'iwylis_vce_Mean', 'mouiwylisv_Mean', 'peak_vce_Mean', 'peak_dat_Mean',
       'mou_peav_Mean', 'mou_pead_Mean', 'opk_vce_Mean', '

In [56]:
# 3. Target
target = 'hnd_price'
print(df[target].value_counts(normalize=True))

hnd_price
149,9899902    0.222232
29,98999023    0.219227
129,9899902    0.135276
199,9899902    0.103366
79,98999023    0.096477
59,98999023    0.087652
99,98999023    0.077991
9,989997864    0.043942
39,98999023    0.006626
249,9899902    0.002461
399,9899902    0.001926
299,9899902    0.000938
179,9899902    0.000797
499,9899902    0.000635
239,9899902    0.000424
159,9899902    0.000020
119,9899902    0.000010
Name: proportion, dtype: float64


In [21]:
# 4. Dimensioni e qualità
print(df.shape)
print(df.isna().sum().sort_values(ascending=False).head(20)) # missing per colonna
print(df.describe(include='all').T) # panoramica statistiche base

(100000, 100)
numbcars            49366
dwllsize            38308
HHstatin            37923
ownrent             33706
dwlltype            31909
lor                 30190
income              25436
adults              23019
infobase            22079
hnd_webcap          10189
prizm_social_one     7388
avg6rev              2839
avg6mou              2839
avg6qty              2839
ethnic               1732
rv                   1732
marital              1732
forgntvl             1732
kid0_2               1732
kid3_5               1732
dtype: int64
                count unique    top   freq        mean           std  \
rev_Mean        99643  37468  29,99   2109         NaN           NaN   
mou_Mean        99643   9730      0   1615         NaN           NaN   
totmrc_Mean     99643   8491  44,99  11192         NaN           NaN   
da_Mean         99643    172      0  48411         NaN           NaN   
ovrmou_Mean     99643   2638      0  42603         NaN           NaN   
...               ...

In [ ]:
# 5. Converti colonne numeriche con virgola decimale
exclude = [
    'churn', 'Customer_ID',
    'new_cell', 'crclscod', 'asl_flag',
    'prizm_social_one', 'area', 'dualband', 'refurb_new',
    'phones', 'models', 'hnd_webcap', 'truck', 'rv',
    'ownrent', 'dwlltype', 'marital', 'infobase',
    'numbcars', 'HHstatin', 'dwllsize', 'forgntvl', 'ethnic',
    'kid0_2', 'kid3_5', 'kid6_10', 'kid11_15', 'kid16_17',
    'creditcd'
]

num_cols = [c for c in df.columns if c not in exclude]

for col in num_cols:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
# 6. Ricontrolla tipi
print(df.dtypes.value_counts())

float64    69
object     21
int64      10
Name: count, dtype: int64


In [59]:
# 7. Gestione missing
missing_pct = df.isna().mean().sort_values(ascending=False)*100
print(missing_pct.head(20))

numbcars            49.366
dwllsize            38.308
HHstatin            37.923
ownrent             33.706
dwlltype            31.909
lor                 30.190
income              25.436
adults              23.019
infobase            22.079
hnd_webcap          10.189
prizm_social_one     7.388
avg6rev              2.839
avg6mou              2.839
avg6qty              2.839
ethnic               1.732
rv                   1.732
marital              1.732
forgntvl             1.732
kid0_2               1.732
kid3_5               1.732
dtype: float64


In [60]:
# 3a. Drop colonne con troppi missing (>35%)
cols_to_drop = missing_pct[missing_pct > 35].index.tolist()
print("Drop:", cols_to_drop)
df = df.drop(columns=cols_to_drop)

# 3b. Imputazione categoriche rimanenti con "Missing"
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna('Missing')

# 3c. Imputazione numeriche rimanenti con mediana
num_remaining = df.select_dtypes(include='number').columns
for col in num_remaining:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Verifica Finale
print(df.isna().sum().sum()) # deve essere 0

Drop: ['numbcars', 'dwllsize', 'HHstatin']
0
